# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [ ]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: completa a partir de la imagen
        self.start = (0, 0)
        self.walls = {(1,1), (0,3), (2,4), (4,2)}
        self.slippery_states = {(1,2), (2,1), (3,3)}

        self.terminal_states = {
            # (row, col): reward
            (0,5): 10,
            (2,2): 2,
            (3,5): -10
        }

        self.danger_states = {
            # (row, col): -3
            (4,1): -3,
            (1,4): -3
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        # TODO
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls


        

    def states(self):
        # TODO
        states = []
        for row in range(self.height):
            for col in range(self.width):
                state = (row, col)
                if self.is_valid_state(state):
                    states.append(state)
        return states

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s)
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_next_state(self, state, action):
        """
        Devuelve el estado resultante de aplicar 'action' en 'state'.
        Si golpea pared/borde, next_state = state.
        """
        # TODO
        row, col = state
        new_row = row + action[0]
        new_col = col + action[1]
        next_state = (new_row, new_col)
        if self.is_valid_state(next_state):
            return next_state
        else:
            return state

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # TODO
        row, col = state
        if self.is_valid_state(state):
            intended_move = action
            left_move = (-action[1], action[0])  # rotar 90 grados a la izquierda
            right_move = (action[1], -action[0])  # rotar 90 grados a la derecha
            if state in self.slippery_states:

                next_states = [
                    (self.get_next_state(state, intended_move), 0.6),
                    (self.get_next_state(state, left_move), 0.2),
                    (self.get_next_state(state, right_move), 0.2)
                ]
            elif state not in self.slippery_states:

                next_states = [
                    (self.get_next_state(state, action), 0.9),
                    (self.get_next_state(state, left_move), 0.05),
                    (self.get_next_state(state, right_move), 0.05)

                ]
        return next_states



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [24]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        print(f"Estado: {s}, Acción: {a}, Transiciones: {transitions}")
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")
print("estados: ", S)


Número de estados: 26
Estado: (0, 0), Acción: (-1, 0), Transiciones: [((0, 0), 0.9), ((0, 0), 0.05), ((0, 1), 0.05)]
Estado: (0, 0), Acción: (1, 0), Transiciones: [((1, 0), 0.9), ((0, 1), 0.05), ((0, 0), 0.05)]
Estado: (0, 0), Acción: (0, -1), Transiciones: [((0, 0), 0.9), ((1, 0), 0.05), ((0, 0), 0.05)]
Estado: (0, 0), Acción: (0, 1), Transiciones: [((0, 1), 0.9), ((0, 0), 0.05), ((1, 0), 0.05)]
Estado: (0, 1), Acción: (-1, 0), Transiciones: [((0, 1), 0.9), ((0, 0), 0.05), ((0, 2), 0.05)]
Estado: (0, 1), Acción: (1, 0), Transiciones: [((0, 1), 0.9), ((0, 2), 0.05), ((0, 0), 0.05)]
Estado: (0, 1), Acción: (0, -1), Transiciones: [((0, 0), 0.9), ((0, 1), 0.05), ((0, 1), 0.05)]
Estado: (0, 1), Acción: (0, 1), Transiciones: [((0, 2), 0.9), ((0, 1), 0.05), ((0, 1), 0.05)]
Estado: (0, 2), Acción: (-1, 0), Transiciones: [((0, 2), 0.9), ((0, 1), 0.05), ((0, 2), 0.05)]
Estado: (0, 2), Acción: (1, 0), Transiciones: [((1, 2), 0.9), ((0, 2), 0.05), ((0, 1), 0.05)]
Estado: (0, 2), Acción: (0, -1), 


## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    total = 0.0
    for next_state, prob in grid.get_transition_probs(state, action):
        total += prob * V[next_state]
    return total


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # V_{k+1}(s) = R(s) + gamma * max_a sum_{s'} T(s,a,s') V_k(s')
    V = {s: 0.0 for s in grid.states()}

    for k in range(1, max_iter + 1):
        V_new = {}
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                # en un terminal no hay acciones: V(s) = R(s)
                V_new[s] = grid.get_reward(s)
            else:
                best = max(expected_next_value(grid, s, a, V) for a in grid.actions)
                V_new[s] = grid.get_reward(s) + grid.gamma * best

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < threshold:
            return V, k

    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, s, a, V)
        )
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # TODO
    pass


def policy_improvement(grid, V):
    # TODO
    pass


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # TODO:
    # 1. política inicial arbitraria
    # 2. evaluación
    # 3. mejora
    # 4. repetir hasta estabilidad
    pass



## Parte 4 — Visualización y comparación


In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")



## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.
